> ## SOLUTION / ANSWER KEY &mdash; Lab 6.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-langgraph-state-machine.ipynb`](../lab-08-langgraph-state-machine.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.8 &mdash; LangGraph: a State Graph with Human Approval

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Route each step to a node: tool, human-approval, or end
- Wire a real LangGraph StateGraph with conditional edges
- Run a sequence through the compiled graph and read the node path

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; LangGraph -- agents as state graphs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-08"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
When one implicit loop isn't enough, **LangGraph** models the agent as a **state graph**: **nodes** are
steps, **edges** are transitions you control (deck slides 12&ndash;13). A key win is a **human approval**
node: a **risky** action (send email, delete, pay) is routed to a human before it runs, while safe
actions go straight to the tool node. Here you wire a **real** `StateGraph` with pure-Python nodes, so
its execution is fully deterministic.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class GState(TypedDict):
    actions: list
    path: list

print("nodes we will wire: tool, human, END |", "END sentinel:", END)

## Your Turn
Complete `route` (which node an action goes to) and the conditional edges, then compile and run the graph.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

RISKY = {"send_email", "delete", "pay"}

class GState(TypedDict):
    actions: list
    path: list

def route(state):
    done_so_far = len(state['path'])
    action = state['actions'][done_so_far] if done_so_far < len(state['actions']) else 'done'
    if action == "done":
        return "end"
    if action in RISKY:
        return "human"
    return "tool"          # any safe tool call

def tool_node(state):  return {'path': state['path'] + ['tool']}
def human_node(state): return {'path': state['path'] + ['human']}

def build_graph():
    g = StateGraph(GState)
    g.add_node('tool', tool_node)
    g.add_node('human', human_node)
    edges = {'tool': 'tool', 'human': 'human', 'end': END}
    g.set_conditional_entry_point(route, edges)
    g.add_conditional_edges('tool', route, edges)
    g.add_conditional_edges('human', route, edges)
    return g.compile()

def run_graph(actions):
    return build_graph().invoke({'actions': actions, 'path': []})['path']

try:
    print('search      ->', route({'actions': ['search'], 'path': []}))
    print('send_email  ->', route({'actions': ['send_email'], 'path': []}))
    print('path:', run_graph(['search', 'send_email', 'done']))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a safe action routes to the tool node", lambda: route({"actions": ["search"], "path": []}) == "tool")
expect_true("a risky action routes to human approval", lambda: route({"actions": ["send_email"], "path": []}) == "human")
expect_true("nothing left routes to the end node", lambda: route({"actions": [], "path": []}) == "end")
expect_true("the compiled graph walks the expected node path", lambda: run_graph(["search", "pay", "done"]) == ["tool", "human", "end"] or run_graph(["search", "pay", "done"]) == ["tool", "human"])
expect_true("a risky step lands on the human node", lambda: "human" in run_graph(["search", "delete"]))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
LangChain gets an agent running; LangGraph puts its flow under control -- explicit nodes, branching, and a first-class human-approval pause. Start simple; graduate when you need it.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>